### This notebook basically focus on cleaning and preparing the dataset for training.

### 1. Dataset cleaning

### 1.1 identifying null values in bookings dataset.

In [2]:
# Cell 1: Load Dataset and Basic Information
import pandas as pd
import numpy as np

# Load the bookings dataset
bookings_df = pd.read_csv('../../datasets/bookings_train.csv')

print("=== BOOKINGS DATASET OVERVIEW ===")
print(f"Dataset shape: {bookings_df.shape}")
print(f"Columns: {list(bookings_df.columns)}")
print("\nDataset Info:")
print(bookings_df.info())
print("\nFirst 5 rows:")
print(bookings_df.head())


=== BOOKINGS DATASET OVERVIEW ===
Dataset shape: (203693, 11)
Columns: ['booking_id', 'citizen_id', 'booking_date', 'appointment_date', 'appointment_time', 'check_in_time', 'check_out_time', 'task_id', 'num_documents', 'queue_number', 'satisfaction_rating']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203693 entries, 0 to 203692
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   booking_id           203693 non-null  object
 1   citizen_id           203693 non-null  int64 
 2   booking_date         203693 non-null  object
 3   appointment_date     203693 non-null  object
 4   appointment_time     203693 non-null  object
 5   check_in_time        197601 non-null  object
 6   check_out_time       197601 non-null  object
 7   task_id              203693 non-null  object
 8   num_documents        203693 non-null  int64 
 9   queue_number         203693 non-null  int64 
 10  satisfactio

In [3]:
# Cell 2: Dataset Cleanliness Check
print("=== DATASET CLEANLINESS ANALYSIS ===")

# Check for missing values
print("\n1. Missing Values:")
missing_values = bookings_df.isnull().sum()
print(missing_values)
print(f"Total missing values: {missing_values.sum()}")

# Check for duplicates
print(f"\n2. Duplicate rows: {bookings_df.duplicated().sum()}")

# Check data types (should match the guide expectations)
print("\n3. Data Types:")
print(bookings_df.dtypes)

# Check for expected columns from the guide
expected_cols = ['booking_id', 'citizen_id', 'appointment_date', 'task_id', 'check_in_time', 'check_out_time']
print(f"\n4. Expected columns present: {all(col in bookings_df.columns for col in expected_cols)}")

# Quick statistical summary for key columns
print("\n5. Key Statistics:")
print(f"   - Unique booking_ids: {bookings_df['booking_id'].nunique()}")
print(f"   - Unique citizen_ids: {bookings_df['citizen_id'].nunique()}")
print(f"   - Unique task_ids: {bookings_df['task_id'].nunique()}")
print(f"   - Date range: {bookings_df['appointment_date'].min()} to {bookings_df['appointment_date'].max()}")

print("\n✅ Dataset cleanliness check completed!")


=== DATASET CLEANLINESS ANALYSIS ===

1. Missing Values:
booking_id                0
citizen_id                0
booking_date              0
appointment_date          0
appointment_time          0
check_in_time          6092
check_out_time         6092
task_id                   0
num_documents             0
queue_number              0
satisfaction_rating       0
dtype: int64
Total missing values: 12184

2. Duplicate rows: 0

3. Data Types:
booking_id             object
citizen_id              int64
booking_date           object
appointment_date       object
appointment_time       object
check_in_time          object
check_out_time         object
task_id                object
num_documents           int64
queue_number            int64
satisfaction_rating     int64
dtype: object

4. Expected columns present: True

5. Key Statistics:
   - Unique booking_ids: 203693
   - Unique citizen_ids: 203688
   - Unique task_ids: 19
   - Date range: 2021-01-01 to 2024-12-31

✅ Dataset cleanliness che

### Remove the null rows

In [ ]:
# Cell 4: Additional Cleaning Verification
print("=== ADDITIONAL CLEANING CHECK ===")

# 1. Check for any remaining missing values
print("1. Remaining missing values:")
print(bookings_df.isnull().sum())

# 2. First, let's examine the time data format
print("\n2. Examining time data format:")
print("Sample check_in_time values:")
print(bookings_df['check_in_time'].head(10).tolist())
print("Sample check_out_time values:")
print(bookings_df['check_out_time'].head(10).tolist())

# Clean time data - extract only time part if datetime is present
def clean_time_data(time_str):
    if pd.isna(time_str):
        return time_str
    time_str = str(time_str)
    # If it contains a date, extract just the time part
    if ' ' in time_str and len(time_str) > 10:
        return time_str.split(' ')[-1]  # Get the last part (time)
    return time_str

# Apply cleaning to time columns
bookings_df['check_in_time_clean'] = bookings_df['check_in_time'].apply(clean_time_data)
bookings_df['check_out_time_clean'] = bookings_df['check_out_time'].apply(clean_time_data)

print("\nAfter cleaning - Sample time values:")
print("check_in_time_clean:", bookings_df['check_in_time_clean'].head(5).tolist())
print("check_out_time_clean:", bookings_df['check_out_time_clean'].head(5).tolist())

# Now convert to datetime safely
try:
    bookings_df['check_in_datetime'] = pd.to_datetime(bookings_df['appointment_date'] + ' ' + bookings_df['check_in_time_clean'], errors='coerce')
    bookings_df['check_out_datetime'] = pd.to_datetime(bookings_df['appointment_date'] + ' ' + bookings_df['check_out_time_clean'], errors='coerce')
    
    # Check for conversion failures
    conversion_failures = bookings_df[bookings_df['check_in_datetime'].isna() | bookings_df['check_out_datetime'].isna()]
    print(f"\n3. Datetime conversion failures: {len(conversion_failures)}")
    
    # Calculate processing minutes
    bookings_df['processing_minutes'] = (bookings_df['check_out_datetime'] - bookings_df['check_in_datetime']).dt.total_seconds() / 60
    
    # 4. Check for invalid time logic
    invalid_times = bookings_df[bookings_df['processing_minutes'] <= 0]
    print(f"\n4. Invalid time records (check_out <= check_in): {len(invalid_times)}")
    
    # 5. Check for duplicate booking_ids
    duplicate_bookings = bookings_df['booking_id'].duplicated().sum()
    print(f"\n5. Duplicate booking_ids: {duplicate_bookings}")
    
    # 6. Check for unrealistic processing times (> 8 hours)
    extreme_times = bookings_df[bookings_df['processing_minutes'] > 480]
    print(f"\n6. Extremely long processing times (>8 hours): {len(extreme_times)}")
    if len(extreme_times) > 0:
        print(f"   Max processing time: {bookings_df['processing_minutes'].max():.1f} minutes")
    
    print(f"\n=== CLEANING SUMMARY ===")
    total_issues = len(conversion_failures) + len(invalid_times) + duplicate_bookings + len(extreme_times)
    print(f"Total additional issues found: {total_issues}")
    
    if total_issues == 0:
        print("✅ No additional cleaning required!")
    else:
        print("⚠️  Additional cleaning recommended")
        
except Exception as e:
    print(f"Error in datetime conversion: {e}")
    print("Manual inspection of time data needed")


In [ ]:
# Cell 5: Daily Aggregation - Calculate daily averages
print("=== DAILY AGGREGATION ===")

# Group by appointment_date and calculate daily averages
daily_aggregated = bookings_df.groupby('appointment_date').agg({
    'processing_minutes': ['mean', 'count', 'sum'],
    'num_documents': 'mean',
    'satisfaction_rating': 'mean',
    'booking_id': 'count'  # Total bookings per day
}).round(2)

# Flatten column names
daily_aggregated.columns = [
    'avg_processing_minutes', 'total_bookings', 'total_processing_minutes',
    'avg_num_documents', 'avg_satisfaction_rating', 'booking_count_check'
]

# Drop the duplicate booking count column
daily_aggregated = daily_aggregated.drop('booking_count_check', axis=1)

# Reset index to make date a column
daily_aggregated = daily_aggregated.reset_index()

print(f"Daily aggregated data shape: {daily_aggregated.shape}")
print("\nFirst 10 days of aggregated data:")
print(daily_aggregated.head(10))

print(f"\nDate range: {daily_aggregated['appointment_date'].min()} to {daily_aggregated['appointment_date'].max()}")
print(f"Total days: {len(daily_aggregated)}")

print("\nDaily averages summary:")
print(f"Average bookings per day: {daily_aggregated['total_bookings'].mean():.1f}")
print(f"Average processing time: {daily_aggregated['avg_processing_minutes'].mean():.1f} minutes")
print(f"Average documents per booking: {daily_aggregated['avg_num_documents'].mean():.1f}")
print(f"Average satisfaction rating: {daily_aggregated['avg_satisfaction_rating'].mean():.1f}")

print("\n✅ Daily aggregation completed!")


In [ ]:
# Cell 6: Task-based Aggregation - Calculate statistics by task_id
print("=== TASK-BASED AGGREGATION ===")

# Group by task_id and calculate task-specific averages
task_aggregated = bookings_df.groupby('task_id').agg({
    'processing_minutes': ['mean', 'count', 'sum', 'std'],
    'num_documents': 'mean',
    'satisfaction_rating': 'mean',
    'booking_id': 'count'  # Total bookings per task
}).round(2)

# Flatten column names
task_aggregated.columns = [
    'avg_processing_minutes', 'total_bookings', 'total_processing_minutes', 'std_processing_minutes',
    'avg_num_documents', 'avg_satisfaction_rating', 'booking_count_check'
]

# Drop the duplicate booking count column
task_aggregated = task_aggregated.drop('booking_count_check', axis=1)

# Reset index to make task_id a column
task_aggregated = task_aggregated.reset_index()

# Sort by total bookings (most popular tasks first)
task_aggregated = task_aggregated.sort_values('total_bookings', ascending=False)

print(f"Task aggregated data shape: {task_aggregated.shape}")
print(f"Number of unique tasks: {len(task_aggregated)}")

print("\nTask statistics (sorted by popularity):")
# Display the dataframe nicely formatted
from IPython.display import display
display(task_aggregated)

print("\nTask complexity analysis:")
print(f"Most time-consuming task: {task_aggregated.loc[task_aggregated['avg_processing_minutes'].idxmax(), 'task_id']} "
      f"({task_aggregated['avg_processing_minutes'].max():.1f} min)")
print(f"Fastest task: {task_aggregated.loc[task_aggregated['avg_processing_minutes'].idxmin(), 'task_id']} "
      f"({task_aggregated['avg_processing_minutes'].min():.1f} min)")
print(f"Most popular task: {task_aggregated.iloc[0]['task_id']} "
      f"({task_aggregated.iloc[0]['total_bookings']} bookings)")

print("\nOverall task averages:")
print(f"Average processing time across all tasks: {task_aggregated['avg_processing_minutes'].mean():.1f} minutes")
print(f"Average documents per task: {task_aggregated['avg_num_documents'].mean():.1f}")
print(f"Average satisfaction across all tasks: {task_aggregated['avg_satisfaction_rating'].mean():.1f}")

print("\n✅ Task-based aggregation completed!")


In [ ]:
# Cell 7: Task + Date Aggregation - Calculate statistics by task_id AND appointment_date
print("=== TASK + DATE AGGREGATION ===")

# Group by both task_id and appointment_date to get daily task performance
task_date_aggregated = bookings_df.groupby(['appointment_date', 'task_id']).agg({
    'processing_minutes': ['mean', 'count', 'sum', 'std'],
    'num_documents': 'mean',
    'satisfaction_rating': 'mean',
    'queue_number': 'mean',
    'booking_id': 'count'
}).round(2)

# Flatten column names
task_date_aggregated.columns = [
    'avg_processing_minutes', 'total_bookings', 'total_processing_minutes', 'std_processing_minutes',
    'avg_num_documents', 'avg_satisfaction_rating', 'avg_queue_number', 'booking_count_check'
]

# Drop the duplicate booking count column
task_date_aggregated = task_date_aggregated.drop('booking_count_check', axis=1)

# Reset index to make appointment_date and task_id regular columns
task_date_aggregated = task_date_aggregated.reset_index()

# Sort by date and then by task_id for better readability
task_date_aggregated = task_date_aggregated.sort_values(['appointment_date', 'task_id'])

print(f"Task+Date aggregated data shape: {task_date_aggregated.shape}")
print(f"Date range: {task_date_aggregated['appointment_date'].min()} to {task_date_aggregated['appointment_date'].max()}")
print(f"Unique tasks: {task_date_aggregated['task_id'].nunique()}")

print("\nTask+Date statistics (first 15 rows):")
# Display the dataframe nicely formatted
from IPython.display import display
display(task_date_aggregated.head(15))

print("\nSample analysis:")
print(f"Total task-date combinations: {len(task_date_aggregated)}")
print(f"Average bookings per task per day: {task_date_aggregated['total_bookings'].mean():.1f}")
print(f"Most busy task-date combination: {task_date_aggregated.loc[task_date_aggregated['total_bookings'].idxmax()]['appointment_date']} - {task_date_aggregated.loc[task_date_aggregated['total_bookings'].idxmax()]['task_id']} ({task_date_aggregated['total_bookings'].max()} bookings)")

# Show sample of specific date to see all tasks on that day
sample_date = task_date_aggregated['appointment_date'].iloc[0]
print(f"\nExample: All tasks on {sample_date}:")
display(task_date_aggregated[task_date_aggregated['appointment_date'] == sample_date])

print("\n✅ Task+Date aggregation completed!")


In [ ]:
# Cell 8: Load Tasks Dataset and Join with Section Information
print("=== JOINING WITH TASKS DATASET FOR SECTION INFORMATION ===")

# Load the tasks dataset
tasks_df = pd.read_csv('../../datasets/tasks.csv')

print("Tasks dataset overview:")
print(f"Shape: {tasks_df.shape}")
print(f"Columns: {list(tasks_df.columns)}")
print("\nFirst 10 rows of tasks dataset:")
display(tasks_df.head(10))

print(f"\nUnique sections in tasks dataset: {tasks_df['section_id'].nunique()}")
print(f"Section IDs: {sorted(tasks_df['section_id'].unique())}")

# Join the task+date aggregated data with tasks to get section information
task_date_with_sections = task_date_aggregated.merge(
    tasks_df[['task_id', 'section_id']], 
    on='task_id', 
    how='left'
)

print(f"\nAfter joining with tasks dataset:")
print(f"Shape: {task_date_with_sections.shape}")
print(f"Columns: {list(task_date_with_sections.columns)}")

# Reorder columns to have section_id right after task_id
cols = task_date_with_sections.columns.tolist()
cols.remove('section_id')
cols.insert(2, 'section_id')  # Insert section_id as 3rd column
task_date_with_sections = task_date_with_sections[cols]

print("\nTask+Date+Section data (first 15 rows):")
display(task_date_with_sections.head(15))

# Verify the join worked correctly
print(f"\nJoin verification:")
print(f"Rows before join: {len(task_date_aggregated)}")
print(f"Rows after join: {len(task_date_with_sections)}")
print(f"Missing section_ids: {task_date_with_sections['section_id'].isnull().sum()}")

# Show distribution of tasks by section
print(f"\nTasks per section:")
section_task_counts = task_date_with_sections.groupby('section_id')['task_id'].nunique().sort_values(ascending=False)
print(section_task_counts)

print("\n✅ Successfully joined with section information!")


In [ ]:
# Cell 9: Aggregate Daily Statistics Per Section
print("=== AGGREGATE DAILY STATISTICS PER SECTION ===")
print("Group bookings by date and section to create daily summaries:")

# Aggregate by date and section_id to create daily section summaries
daily_section_summary = task_date_with_sections.groupby(['appointment_date', 'section_id']).agg({
    'total_bookings': 'sum',  # Total bookings per section per day
    'avg_processing_minutes': 'mean',  # Average processing time per section per day
    'total_processing_minutes': 'sum',  # Total processing time per section per day
    'avg_num_documents': 'mean',  # Average documents per section per day
    'avg_satisfaction_rating': 'mean',  # Average satisfaction per section per day
    'avg_queue_number': 'mean'  # Average queue number per section per day
}).round(2)

# Flatten column names and rename to match the screenshot format
daily_section_summary.columns = [
    'num_bookings', 'avg_processing_time', 'total_processing', 
    'avg_documents', 'avg_satisfaction', 'avg_queue_number'
]

# Reset index to make date and section_id regular columns
daily_section_summary = daily_section_summary.reset_index()

# Rename columns to match screenshot exactly
daily_section_summary = daily_section_summary.rename(columns={
    'appointment_date': 'date',
    'avg_processing_time': 'avg_processing_time',
    'total_processing': 'total_processing'
})

# Sort by date and section for better readability
daily_section_summary = daily_section_summary.sort_values(['date', 'section_id'])

print(f"\nDaily Section Summary shape: {daily_section_summary.shape}")
print(f"Date range: {daily_section_summary['date'].min()} to {daily_section_summary['date'].max()}")
print(f"Unique sections: {daily_section_summary['section_id'].nunique()}")

print("\nAggregation Operations:")
print("• num_bookings: COUNT of bookings per section per day")
print("• avg_processing_time: AVERAGE processing time")
print("• total_processing: SUM of all processing times")
print("• avg_documents: AVERAGE documents per booking")
print("• avg_satisfaction: AVERAGE satisfaction rating")
print("• avg_queue_number: AVERAGE queue position")

print("\nDaily Section Summary (first 15 rows):")
display(daily_section_summary.head(15))

# Show example matching the screenshot format
print(f"\nExample: All sections on {daily_section_summary['date'].iloc[0]}:")
sample_date = daily_section_summary['date'].iloc[0]
sample_data = daily_section_summary[daily_section_summary['date'] == sample_date][['date', 'section_id', 'num_bookings', 'avg_processing_time', 'total_processing']]
display(sample_data)

print(f"\nSummary Statistics:")
print(f"Average bookings per section per day: {daily_section_summary['num_bookings'].mean():.1f}")
print(f"Average processing time across sections: {daily_section_summary['avg_processing_time'].mean():.1f} minutes")
print(f"Total daily section-date combinations: {len(daily_section_summary)}")

print("\n✅ Daily section aggregation completed!")


In [4]:
# Cell 3: Data Cleaning - Remove rows with missing check-in/check-out times
print("=== DATA CLEANING ===")

print(f"Original dataset size: {len(bookings_df):,} rows")

# Remove rows where check_in_time or check_out_time is missing
# These are essential for calculating processing_minutes
bookings_clean = bookings_df.dropna(subset=['check_in_time', 'check_out_time'])

print(f"After removing missing check-in/check-out: {len(bookings_clean):,} rows")
print(f"Rows removed: {len(bookings_df) - len(bookings_clean):,}")
print(f"Data retention: {(len(bookings_clean) / len(bookings_df)) * 100:.1f}%")

# Verify no missing values in critical columns
print(f"\nMissing values after cleaning:")
print(bookings_clean[['check_in_time', 'check_out_time']].isnull().sum())

# Update our working dataset
bookings_df = bookings_clean.copy()
print(f"\n✅ Clean dataset ready: {len(bookings_df):,} rows with complete check-in/check-out data")


=== DATA CLEANING ===
Original dataset size: 203,693 rows
After removing missing check-in/check-out: 197,601 rows
Rows removed: 6,092
Data retention: 97.0%

Missing values after cleaning:
check_in_time     0
check_out_time    0
dtype: int64

✅ Clean dataset ready: 197,601 rows with complete check-in/check-out data


In [7]:


# Cell 4: Additional Cleaning Verification
print("=== ADDITIONAL CLEANING CHECK ===")

# 1. Check for any remaining missing values
print("1. Remaining missing values:")
print(bookings_df.isnull().sum())

# 2. First, let's examine the time data format
print("\n2. Examining time data format:")
print("Sample check_in_time values:")
print(bookings_df['check_in_time'].head(10).tolist())
print("Sample check_out_time values:")
print(bookings_df['check_out_time'].head(10).tolist())

# Clean time data - extract only time part if datetime is present
def clean_time_data(time_str):
    if pd.isna(time_str):
        return time_str
    time_str = str(time_str)
    # If it contains a date, extract just the time part
    if ' ' in time_str and len(time_str) > 10:
        return time_str.split(' ')[-1]  # Get the last part (time)
    return time_str

# Apply cleaning to time columns
bookings_df['check_in_time_clean'] = bookings_df['check_in_time'].apply(clean_time_data)
bookings_df['check_out_time_clean'] = bookings_df['check_out_time'].apply(clean_time_data)

print("\nAfter cleaning - Sample time values:")
print("check_in_time_clean:", bookings_df['check_in_time_clean'].head(5).tolist())
print("check_out_time_clean:", bookings_df['check_out_time_clean'].head(5).tolist())

# Now convert to datetime safely
try:
    bookings_df['check_in_datetime'] = pd.to_datetime(bookings_df['appointment_date'] + ' ' + bookings_df['check_in_time_clean'], errors='coerce')
    bookings_df['check_out_datetime'] = pd.to_datetime(bookings_df['appointment_date'] + ' ' + bookings_df['check_out_time_clean'], errors='coerce')
    
    # Check for conversion failures
    conversion_failures = bookings_df[bookings_df['check_in_datetime'].isna() | bookings_df['check_out_datetime'].isna()]
    print(f"\n3. Datetime conversion failures: {len(conversion_failures)}")
    
    # Calculate processing minutes
    bookings_df['processing_minutes'] = (bookings_df['check_out_datetime'] - bookings_df['check_in_datetime']).dt.total_seconds() / 60
    
    # 4. Check for invalid time logic
    invalid_times = bookings_df[bookings_df['processing_minutes'] <= 0]
    print(f"\n4. Invalid time records (check_out <= check_in): {len(invalid_times)}")
    
    # 5. Check for duplicate booking_ids
    duplicate_bookings = bookings_df['booking_id'].duplicated().sum()
    print(f"\n5. Duplicate booking_ids: {duplicate_bookings}")
    
    # 6. Check for unrealistic processing times (> 8 hours)
    extreme_times = bookings_df[bookings_df['processing_minutes'] > 480]
    print(f"\n6. Extremely long processing times (>8 hours): {len(extreme_times)}")
    if len(extreme_times) > 0:
        print(f"   Max processing time: {bookings_df['processing_minutes'].max():.1f} minutes")
    
    print(f"\n=== CLEANING SUMMARY ===")
    total_issues = len(conversion_failures) + len(invalid_times) + duplicate_bookings + len(extreme_times)
    print(f"Total additional issues found: {total_issues}")
    
    if total_issues == 0:
        print("✅ No additional cleaning required!")
    else:
        print("⚠️  Additional cleaning recommended")
        
except Exception as e:
    print(f"Error in datetime conversion: {e}")
    print("Manual inspection of time data needed")

=== ADDITIONAL CLEANING CHECK ===
1. Remaining missing values:
booking_id             0
citizen_id             0
booking_date           0
appointment_date       0
appointment_time       0
check_in_time          0
check_out_time         0
task_id                0
num_documents          0
queue_number           0
satisfaction_rating    0
dtype: int64

2. Examining time data format:
Sample check_in_time values:
['2021-01-01 09:11:00', '2021-01-01 09:24:00', '2021-01-01 09:29:00', '2021-01-01 10:07:00', '2021-01-01 10:26:00', '2021-01-01 10:25:00', '2021-01-01 10:29:00', '2021-01-01 10:35:00', '2021-01-01 10:34:00', '2021-01-01 13:54:00']
Sample check_out_time values:
['2021-01-01 09:48:15.166353269', '2021-01-01 10:24:12.189261137', '2021-01-01 10:26:48.802260864', '2021-01-01 11:00:13.485642822', '2021-01-01 11:54:53.260180213', '2021-01-01 12:06:10.680457034', '2021-01-01 11:45:37.549181971', '2021-01-01 11:33:33.010326653', '2021-01-01 11:34:08.857731647', '2021-01-01 14:41:52.68746630

#### 2. Aggregating all the values for one day (we are taking the average of other values here)

In [11]:

# Cell 7: Task + Date Aggregation - Calculate statistics by task_id AND appointment_date
print("=== TASK + DATE AGGREGATION ===")

# Group by both task_id and appointment_date to get daily task performance
task_date_aggregated = bookings_df.groupby(['appointment_date', 'task_id']).agg({
    'processing_minutes': ['mean', 'count', 'sum', 'std'],
    'num_documents': 'mean',
    'satisfaction_rating': 'mean',
    'queue_number': 'mean',
    'booking_id': 'count'
}).round(2)

# Flatten column names
task_date_aggregated.columns = [
    'avg_processing_minutes', 'total_bookings', 'total_processing_minutes', 'std_processing_minutes',
    'avg_num_documents', 'avg_satisfaction_rating', 'avg_queue_number', 'booking_count_check'
]

# Drop the duplicate booking count column
task_date_aggregated = task_date_aggregated.drop('booking_count_check', axis=1)

# Reset index to make appointment_date and task_id regular columns
task_date_aggregated = task_date_aggregated.reset_index()

# Sort by date and then by task_id for better readability
task_date_aggregated = task_date_aggregated.sort_values(['appointment_date', 'task_id'])

print(f"Task+Date aggregated data shape: {task_date_aggregated.shape}")
print(f"Date range: {task_date_aggregated['appointment_date'].min()} to {task_date_aggregated['appointment_date'].max()}")
print(f"Unique tasks: {task_date_aggregated['task_id'].nunique()}")

print("\nTask+Date statistics (first 15 rows):")
# Display the dataframe nicely formatted
from IPython.display import display
display(task_date_aggregated.head(15))

print("\nSample analysis:")
print(f"Total task-date combinations: {len(task_date_aggregated)}")
print(f"Average bookings per task per day: {task_date_aggregated['total_bookings'].mean():.1f}")
print(f"Most busy task-date combination: {task_date_aggregated.loc[task_date_aggregated['total_bookings'].idxmax()]['appointment_date']} - {task_date_aggregated.loc[task_date_aggregated['total_bookings'].idxmax()]['task_id']} ({task_date_aggregated['total_bookings'].max()} bookings)")

# Show sample of specific date to see all tasks on that day
sample_date = task_date_aggregated['appointment_date'].iloc[0]
print(f"\nExample: All tasks on {sample_date}:")
display(task_date_aggregated[task_date_aggregated['appointment_date'] == sample_date])

print("\n✅ Task+Date aggregation completed!")

=== TASK + DATE AGGREGATION ===
Task+Date aggregated data shape: (18078, 9)
Date range: 2021-01-01 to 2024-12-31
Unique tasks: 19

Task+Date statistics (first 15 rows):


,appointment_date,task_id,avg_processing_minutes,total_bookings,total_processing_minutes,std_processing_minutes,avg_num_documents,avg_satisfaction_rating,avg_queue_number
0,2021-01-01,TASK-001,58.03,4,232.13,3.29,1.25,4.25,2.25
1,2021-01-01,TASK-002,75.31,8,602.44,26.02,1.38,4.12,5.25
2,2021-01-01,TASK-003,54.20,9,487.79,9.22,4.89,4.33,4.33
3,2021-01-01,TASK-004,27.39,2,54.77,8.59,2.00,4.00,1.50
4,2021-01-01,TASK-005,36.83,10,368.29,6.52,2.40,4.30,6.10
5,2021-01-01,TASK-006,43.89,14,614.51,18.31,2.43,4.43,7.50
6,2021-01-01,TASK-007,22.35,5,111.76,7.57,2.00,4.00,3.00
7,2021-01-01,TASK-008,28.70,8,229.61,6.79,1.88,4.00,5.00
8,2021-01-01,TASK-009,40.09,8,320.73,5.40,3.25,4.12,4.00
9,2021-01-01,TASK-010,42.81,9,385.27,10.83,2.33,4.33,5.00



Sample analysis:
Total task-date combinations: 18078
Average bookings per task per day: 10.9
Most busy task-date combination: 2022-08-08 - TASK-014 (126 bookings)

Example: All tasks on 2021-01-01:


,appointment_date,task_id,avg_processing_minutes,total_bookings,total_processing_minutes,std_processing_minutes,avg_num_documents,avg_satisfaction_rating,avg_queue_number
0,2021-01-01,TASK-001,58.03,4,232.13,3.29,1.25,4.25,2.25
1,2021-01-01,TASK-002,75.31,8,602.44,26.02,1.38,4.12,5.25
2,2021-01-01,TASK-003,54.20,9,487.79,9.22,4.89,4.33,4.33
3,2021-01-01,TASK-004,27.39,2,54.77,8.59,2.00,4.00,1.50
4,2021-01-01,TASK-005,36.83,10,368.29,6.52,2.40,4.30,6.10
5,2021-01-01,TASK-006,43.89,14,614.51,18.31,2.43,4.43,7.50
6,2021-01-01,TASK-007,22.35,5,111.76,7.57,2.00,4.00,3.00
7,2021-01-01,TASK-008,28.70,8,229.61,6.79,1.88,4.00,5.00
8,2021-01-01,TASK-009,40.09,8,320.73,5.40,3.25,4.12,4.00
9,2021-01-01,TASK-010,42.81,9,385.27,10.83,2.33,4.33,5.00



✅ Task+Date aggregation completed!


### Joining with Section ID

In [13]:


# Cell 8: Load Tasks Dataset and Join with Section Information
print("=== JOINING WITH TASKS DATASET FOR SECTION INFORMATION ===")

# Load the tasks dataset
tasks_df = pd.read_csv('../../datasets/tasks.csv')

print("Tasks dataset overview:")
print(f"Shape: {tasks_df.shape}")
print(f"Columns: {list(tasks_df.columns)}")
print("\nFirst 10 rows of tasks dataset:")
display(tasks_df.head(10))

print(f"\nUnique sections in tasks dataset: {tasks_df['section_id'].nunique()}")
print(f"Section IDs: {sorted(tasks_df['section_id'].unique())}")

# Join the task+date aggregated data with tasks to get section information
task_date_with_sections = task_date_aggregated.merge(
    tasks_df[['task_id', 'section_id']], 
    on='task_id', 
    how='left'
)

print(f"\nAfter joining with tasks dataset:")
print(f"Shape: {task_date_with_sections.shape}")
print(f"Columns: {list(task_date_with_sections.columns)}")

# Reorder columns to have section_id right after task_id
cols = task_date_with_sections.columns.tolist()
cols.remove('section_id')
cols.insert(2, 'section_id')  # Insert section_id as 3rd column
task_date_with_sections = task_date_with_sections[cols]

print("\nTask+Date+Section data (first 15 rows):")
display(task_date_with_sections.head(30))

# Verify the join worked correctly
print(f"\nJoin verification:")
print(f"Rows before join: {len(task_date_aggregated)}")
print(f"Rows after join: {len(task_date_with_sections)}")
print(f"Missing section_ids: {task_date_with_sections['section_id'].isnull().sum()}")

# Show distribution of tasks by section
print(f"\nTasks per section:")
section_task_counts = task_date_with_sections.groupby('section_id')['task_id'].nunique().sort_values(ascending=False)
print(section_task_counts)

print("\n✅ Successfully joined with section information!")

=== JOINING WITH TASKS DATASET FOR SECTION INFORMATION ===
Tasks dataset overview:
Shape: (19, 4)
Columns: ['task_id', 'task_name', 'section_id', 'section_name']

First 10 rows of tasks dataset:


,task_id,task_name,section_id,section_name
0,TASK-001,NaN,SEC-001,NaN
1,TASK-002,NaN,SEC-001,NaN
2,TASK-003,NaN,SEC-002,NaN
3,TASK-004,NaN,SEC-002,NaN
4,TASK-005,NaN,SEC-002,NaN
5,TASK-006,NaN,SEC-002,NaN
6,TASK-007,NaN,SEC-003,NaN
7,TASK-008,NaN,SEC-003,NaN
8,TASK-009,NaN,SEC-003,NaN
9,TASK-010,NaN,SEC-004,NaN



Unique sections in tasks dataset: 6
Section IDs: ['SEC-001', 'SEC-002', 'SEC-003', 'SEC-004', 'SEC-005', 'SEC-006']

After joining with tasks dataset:
Shape: (18078, 10)
Columns: ['appointment_date', 'task_id', 'avg_processing_minutes', 'total_bookings', 'total_processing_minutes', 'std_processing_minutes', 'avg_num_documents', 'avg_satisfaction_rating', 'avg_queue_number', 'section_id']

Task+Date+Section data (first 15 rows):


,appointment_date,task_id,section_id,avg_processing_minutes,total_bookings,total_processing_minutes,std_processing_minutes,avg_num_documents,avg_satisfaction_rating,avg_queue_number
0,2021-01-01,TASK-001,SEC-001,58.03,4,232.13,3.29,1.25,4.25,2.25
1,2021-01-01,TASK-002,SEC-001,75.31,8,602.44,26.02,1.38,4.12,5.25
2,2021-01-01,TASK-003,SEC-002,54.20,9,487.79,9.22,4.89,4.33,4.33
3,2021-01-01,TASK-004,SEC-002,27.39,2,54.77,8.59,2.00,4.00,1.50
4,2021-01-01,TASK-005,SEC-002,36.83,10,368.29,6.52,2.40,4.30,6.10
5,2021-01-01,TASK-006,SEC-002,43.89,14,614.51,18.31,2.43,4.43,7.50
6,2021-01-01,TASK-007,SEC-003,22.35,5,111.76,7.57,2.00,4.00,3.00
7,2021-01-01,TASK-008,SEC-003,28.70,8,229.61,6.79,1.88,4.00,5.00
8,2021-01-01,TASK-009,SEC-003,40.09,8,320.73,5.40,3.25,4.12,4.00
9,2021-01-01,TASK-010,SEC-004,42.81,9,385.27,10.83,2.33,4.33,5.00



Join verification:
Rows before join: 18078
Rows after join: 18078
Missing section_ids: 0

Tasks per section:
section_id
SEC-002    4
SEC-006    4
SEC-004    3
SEC-003    3
SEC-005    3
SEC-001    2
Name: task_id, dtype: int64

✅ Successfully joined with section information!
